In [1]:
# ===========================================================
# CELLA 0 — Bootstrap one-shot (eseguire solo prima volta o
# dopo restart Compute Instance). Saltabile nelle sessioni successive.
# ===========================================================
import os, sys, shutil
from pathlib import Path

# ── 1. Verifica dipendenze Python ──────────────────────────────
print("📦 Verifica dipendenze:")
required = ['torch', 'numpy', 'pandas', 'matplotlib']
missing  = []
for pkg in required:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, '__version__', '?')
        print(f"   ✅ {pkg:<12} v{ver}")
    except ImportError:
        print(f"   ❌ {pkg:<12} MANCANTE")
        missing.append(pkg)

if missing:
    print(f"\n⚠️  Installa con: !pip install {' '.join(missing)}")
else:
    print("   → tutte le dipendenze ok\n")

# ── 2. Stato del repo (commit attuale) ─────────────────────────
print("📍 Stato repo:")
!git log --oneline -5
print()
!git status --short

# ── 3. Verifica .gitignore whitelist (deve esistere dal commit bdc1a00) ──
print("\n🔍 Verifica .gitignore whitelist results/:")
with open('.gitignore') as f:
    content = f.read()
if "!results/" in content:
    print("   ✅ Whitelist presente (commit bdc1a00 o successivo)")
else:
    print("   ⚠️  Whitelist MANCANTE — fai git pull origin main")

# ── 4. Stato checkpoints (cosa c'è già) ────────────────────────
print("\n📂 checkpoints/ presenti:")
ckpt_dirs = sorted(os.listdir('checkpoints')) if os.path.isdir('checkpoints') else []
if not ckpt_dirs:
    print("   (vuoto)")
else:
    total_size_mb = 0
    for d in ckpt_dirs:
        size = sum(os.path.getsize(os.path.join(dp, f))
                   for dp, _, fns in os.walk(f"checkpoints/{d}") for f in fns) / 1e6
        total_size_mb += size
        print(f"   • {d:<40} {size:>8.1f} MB")
    print(f"   {'TOTALE':<40} {total_size_mb:>8.1f} MB")

# ── 5. Stato results/ (già pushato a git) ─────────────────────
print("\n📊 results/ presenti (tracked git):")
res_dirs = sorted(os.listdir('results')) if os.path.isdir('results') else []
if not res_dirs:
    print("   (vuoto)")
else:
    for d in res_dirs:
        n_plots = len([f for f in os.listdir(f"results/{d}/plots") if f.endswith('.png')]) \
                  if os.path.isdir(f"results/{d}/plots") else 0
        has_crash = "❌ CRASH" if os.path.isfile(f"results/{d}/CRASH_INFO.txt") else "✅"
        print(f"   • {d:<40} {n_plots} plots  {has_crash}")

# ── 6. Cache dataset ──────────────────────────────────────────
cache_files = [f"data/{f}" for f in os.listdir('data')
               if f.endswith('.pt') and f.startswith('cache')] if os.path.isdir('data') else []
print(f"\n💾 Cache dataset:")
if not cache_files:
    print("   (nessuna cache trovata)")
else:
    for c in cache_files:
        size_mb = os.path.getsize(c) / 1e6
        print(f"   • {c:<40} {size_mb:>8.1f} MB")

# ── 7. Pulizia opzionale (commentata di default per sicurezza) ─
print("\n🗑️  Pulizia opzionale (decommenta solo se necessario):")
print("   # Cancella TUTTI i checkpoints (libera storage Azure):")
print("   # !rm -rf checkpoints/*")
print("   #")
print("   # Cancella solo best_model.pt stale (mantiene i CSV/PNG):")
print("   # !find checkpoints -name 'best_model.pt' -delete")
print("   # !find checkpoints -name 'crash_model.pt' -delete")
print("   #")
print("   # Forza rigenerazione cache (se sospetti incoerenze fisiche):")
print("   # !rm -f data/cache_1500.pt")

print("\n✅ Bootstrap completato — procedi con Cella 1.")

📦 Verifica dipendenze:
   ✅ torch        v2.9.1+cu128
   ✅ numpy        v1.23.5
   ✅ pandas       v1.5.3
   ✅ matplotlib   v3.7.1
   → tutte le dipendenze ok

📍 Stato repo:
bf0d8c6 (HEAD -> main, origin/main, origin/HEAD) docs: P_S.md aggiornato con P7+P8+P9 — nuova diagnosi capacity insufficiency
fd8c5bf results: P6_T3_full (2026-05-27 20:28)
a13afb6 feat: P1 B5 - spike-rate regularizer per evitare degenerazione dead network
bb728ec results: P6_T2_full (2026-05-27 16:46)
1eff0b0 fix: P1 A3 - gamma surrogate 0.3 -> 1.0 per ridurre amplificazione BPTT

?? A1_log.csv
?? Testing.ipynb
?? Untitled.ipynb
?? results_A1_onecycle_v3.zip
?? {cache_path}

🔍 Verifica .gitignore whitelist results/:
   ✅ Whitelist presente (commit bdc1a00 o successivo)

📂 checkpoints/ presenti:
   • A1_onecycle                                   0.1 MB
   • A1_onecycle_v3                                0.4 MB
   • A1_onecycle_v3_preflight_1                    1.3 MB
   • A1_onecycle_v3_preflight_2                   

In [ ]:
# ===========================================================
# CELLA 1 — Configurazione esperimento (l'UNICA cella da modificare)
# ===========================================================
# Tutti i parametri esposti come dict CONFIG.
# - scenario_mix e cut_in_ratio sono ora controllabili da qui (P10)
#   → niente più modifiche manuali a config.py su Azure!
# - early_stop_patience attiva l'early stopping (P11)
#   → ferma il training oltre il plateau, evita crash post-saturazione

TAG = "P9_S1_highway_v2"   # ← cambia per ogni esperimento

CONFIG = {
    'epochs':         5,
    'scheduler':      'onecycle',     # 'onecycle' | 'cosine' | 'plateau'
    'max_lr':         2e-3,           # solo onecycle
    'T0':             5,              # solo cosine
    'lr':             1e-3,           # solo plateau
    'seq_len':        50,
    'batch_size':     64,
    # ── Pesi loss PINN ────────────────────────────
    'lambda_data':    1.0,
    'lambda_phys':    0.1,
    'lambda_ou':      0.05,
    'lambda_bc':      1.0,
    # 'lambda_sr':    0.5,            # B5 — default da config.py (LAMBDA_SR)
    'optimizer':      'adam',
    # ── P10: scenario dataset (override da CLI, sicuro su Azure) ──
    'scenario_mix':   'highway',      # 'default' | 'highway' | 'urban' | 'truck' | 'mixed' | 'highway:0.7,urban:0.3'
    'cut_in_ratio':   0.0,            # 0.0 = no cut-in | None = usa CUT_IN_RATIO da config.py | float 0-1
    # ── P11: early stopping (evita training oltre il plateau) ─────
    'early_stop_patience': 2,         # ferma se val_loss non migliora per N epoche. 0 = disabilitato.
    'early_stop_delta':    1e-4,      # soglia minima miglioramento per resettare patience
    # ── Diagnostica ───────────────────────────────
    'max_inf_streak': 20,
}

# Cache: il nome include lo scenario per evitare collisioni cross-esperimento
CACHE_PATH = f"data/cache_1500_{CONFIG['scenario_mix']}_cut{CONFIG['cut_in_ratio']}.pt"

RUN_GIT_PULL    = True
RUN_PREFLIGHT   = True
RUN_FULL        = True
PUSH_RESULTS    = True

print(f"🎯 Esperimento: {TAG}")
print(f"   Scenario:     {CONFIG['scenario_mix']}  (cut_in={CONFIG['cut_in_ratio']})")
print(f"   Cache:        {CACHE_PATH}")
print(f"   Scheduler:    {CONFIG['scheduler']}  (max_lr={CONFIG.get('max_lr','-')})")
print(f"   seq_len:      {CONFIG['seq_len']}  epochs:{CONFIG['epochs']}")
print(f"   Early stop:   patience={CONFIG['early_stop_patience']}, delta={CONFIG['early_stop_delta']}")
print(f"   Preflight:    {'SÌ' if RUN_PREFLIGHT else 'NO'}")
print(f"   FULL train:   {'SÌ' if RUN_FULL else 'NO'}")
print(f"   Push results: {'SÌ' if PUSH_RESULTS else 'NO'}")


In [3]:
# ===========================================================
# CELLA 2 — Imports + sync + sanity checks (universal)
# ===========================================================
import os, sys, glob, shutil, json, subprocess, datetime
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Image, Markdown, HTML

# Sync con remote
if RUN_GIT_PULL:
    print("🔄 git pull origin main...")
    !git pull origin main

# Mostra commit attuale (utile per tracciare quale codice stiamo testando)
print("\n📍 Commit attuale:")
!git log --oneline -3

# Sanity check imports della nostra infrastruttura
print("\n🔧 Sanity check imports:")
!python -c "from train import BatchCSVLogger, _make_batch_row; \
            from utils.plot_diagnostics import load_batch_log, plot_g13_signals_vs_params; \
            from core.network import CF_FSNN_Net; \
            from core.neurons import ALIFCell; \
            print('✅ Tutti gli import OK')"

# Verifica che lo script preflight esista
assert os.path.isfile('scripts/preflight.py'), "scripts/preflight.py NON trovato"
print("✅ scripts/preflight.py presente")

# Verifica costanti chiave dal sorgente (sanity check che la rete sia "sana")
print("\n🔍 Sanity check codice:")
!python -c "import inspect; from core.neurons import ALIFCell; \
            src = inspect.getsource(ALIFCell.forward); \
            assert '.detach()' not in src.split('# Soft Reset')[1].split('return')[0] or 'B4 ROLLBACK' in src, \
                'B4 detach ancora presente nel reset path! Vedi P_S.md sezione P5.'; \
            print('✅ ALIFCell.forward sano (no B4 detach orfano)')"

🔄 git pull origin main...
From https://github.com/carmineesposito01-ice-beep/SNN_Experiment
 * branch            main       -> FETCH_HEAD
Already up to date.

📍 Commit attuale:
bf0d8c6 (HEAD -> main, origin/main, origin/HEAD) docs: P_S.md aggiornato con P7+P8+P9 — nuova diagnosi capacity insufficiency
fd8c5bf results: P6_T3_full (2026-05-27 20:28)
a13afb6 feat: P1 B5 - spike-rate regularizer per evitare degenerazione dead network

🔧 Sanity check imports:
✅ Tutti gli import OK
✅ scripts/preflight.py presente

🔍 Sanity check codice:
✅ ALIFCell.forward sano (no B4 detach orfano)


In [4]:
# ===========================================================
# CELLA 3 — Cache management (universal)
# ===========================================================
os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)

if os.path.exists(CACHE_PATH):
    size_mb = os.path.getsize(CACHE_PATH) / 1e6
    mtime_ts = os.path.getmtime(CACHE_PATH)
    mtime = datetime.datetime.fromtimestamp(mtime_ts).strftime('%Y-%m-%d %H:%M')
    print(f"✅ Cache presente: {CACHE_PATH}")
    print(f"   Size:         {size_mb:.1f} MB")
    print(f"   Modificata:   {mtime}")
    
    # Verifica caricabile
    import torch
    cache = torch.load(CACHE_PATH, weights_only=False)
    print(f"   Train trajs:  {len(cache['train'])}")
    print(f"   Val trajs:    {len(cache['val'])}")
    print(f"   Seed:         {cache.get('seed', 'N/A')}")
    print(f"   → FULL caricherà in <5s")
else:
    print(f"⚠️  Cache assente: {CACHE_PATH}")
    print(f"   → FULL la rigenererà (~30s GPU, ~5min CPU)")

⚠️  Cache assente: data/cache_1500_highway_only.pt
   → FULL la rigenererà (~30s GPU, ~5min CPU)


In [5]:
# ===========================================================
# CELLA 4 — Pre-flight doppio smoke (universal)
# ===========================================================
if not RUN_PREFLIGHT:
    print("⏭️  RUN_PREFLIGHT=False — skipped")
else:
    # Costruisci args extra dal CONFIG (passa seq_len e max_lr al preflight
    # per matchare il config FULL — gli altri parametri lo smoke li ignora)
    extra = ['--seq_len', str(CONFIG['seq_len'])]
    if CONFIG['scheduler'] == 'onecycle' and 'max_lr' in CONFIG:
        extra += ['--max_lr', str(CONFIG['max_lr'])]
    
    cmd = ['python', 'scripts/preflight.py', '--base_tag', TAG, '--extra'] + extra
    print(f"🚦 Eseguo: {' '.join(cmd)}\n" + "=" * 72)
    
    # Subprocess per catturare exit code
    result = subprocess.run(cmd, capture_output=False)
    preflight_ok = (result.returncode == 0)
    
    print("\n" + "=" * 72)
    if preflight_ok:
        print(f"✅ PREFLIGHT PASS — safe per lanciare FULL")
    else:
        print(f"❌ PREFLIGHT FAIL — esit code {result.returncode}")
        print(f"   NON lanciare FULL. Esamina:")
        print(f"     - checkpoints/{TAG}_preflight_1/")
        print(f"     - checkpoints/{TAG}_preflight_2/")

🚦 Eseguo: python scripts/preflight.py --base_tag P9_S1_highway_only --extra --seq_len 50 --max_lr 0.002

[preflight] Base tag: P9_S1_highway_only
[preflight] Smoke 1: P9_S1_highway_only_preflight_1
[preflight] Smoke 2: P9_S1_highway_only_preflight_2
[preflight] Extra args: ['--seq_len', '50', '--max_lr', '0.002']

[preflight] Lancio: /anaconda/envs/azureml_py38/bin/python train.py --smoke --tag P9_S1_highway_only_preflight_1 --seq_len 50 --max_lr 0.002
[SMOKE] Modalita diagnostica: n_train<=100, n_val<=30, 1 epoca, LOG_EVERY=1, max_inf_streak=5, norme per-layer attive

[CF_FSNN] Device: cpu  |  Tag: P9_S1_highway_only_preflight_1  |  SEED: 42
  Save dir: /mnt/batch/tasks/shared/LS_root/mounts/clusters/sandokan/code/SNN_Experiment/checkpoints/P9_S1_highway_only_preflight_1
[Dataset] Generazione sintetica ACC-IDM ...

 STATS: train
 Traiettorie       : 100
 Steps per traj    : 1000
 Scenari           : {'highway': 48, 'urban': 30, 'truck': 9, 'mixed': 13}
 Cut-in            : 22 (22.0%)


In [ ]:
# ===========================================================
# CELLA 5 (P10+P11) — FULL training (universal)
# Supporta scenario_mix, cut_in_ratio, early_stop_patience da CONFIG
# ===========================================================
if not RUN_FULL:
    print("⏭️  RUN_FULL=False — skipped")
elif RUN_PREFLIGHT and not preflight_ok:
    print("⚠️  Preflight FAIL — FULL non lanciato per sicurezza")
    print("   Per forzare comunque: imposta RUN_PREFLIGHT=False in Cella 1 e ri-esegui.")
else:
    cli_args = [
        '--epochs',         str(CONFIG['epochs']),
        '--scheduler',      CONFIG['scheduler'],
        '--seq_len',        str(CONFIG['seq_len']),
        '--batch_size',     str(CONFIG['batch_size']),
        '--lambda_data',    str(CONFIG['lambda_data']),
        '--lambda_phys',    str(CONFIG['lambda_phys']),
        '--lambda_ou',      str(CONFIG['lambda_ou']),
        '--lambda_bc',      str(CONFIG['lambda_bc']),
        '--optimizer',      CONFIG['optimizer'],
        '--max_inf_streak', str(CONFIG['max_inf_streak']),
        '--data_cache',     CACHE_PATH,
        '--tag',            TAG,
        # P10: scenario override
        '--scenario_mix',   CONFIG['scenario_mix'],
        # P11: early stopping
        '--early_stop_patience', str(CONFIG['early_stop_patience']),
        '--early_stop_delta',    str(CONFIG['early_stop_delta']),
    ]
    # cut_in_ratio: solo se specificato (None → usa default config.py)
    if CONFIG.get('cut_in_ratio') is not None:
        cli_args += ['--cut_in_ratio', str(CONFIG['cut_in_ratio'])]
    # B5: lambda_sr opzionale (default da config.py)
    if 'lambda_sr' in CONFIG:
        cli_args += ['--lambda_sr', str(CONFIG['lambda_sr'])]
    # Parametri specifici dello scheduler
    if CONFIG['scheduler'] == 'onecycle':
        cli_args += ['--max_lr', str(CONFIG['max_lr'])]
    elif CONFIG['scheduler'] == 'cosine':
        cli_args += ['--T0', str(CONFIG['T0'])]
    elif CONFIG['scheduler'] == 'plateau':
        cli_args += ['--lr', str(CONFIG['lr'])]

    cmd = ['python', 'train.py'] + cli_args
    print(f"🚀 Eseguo: {' '.join(cmd)}\n" + "=" * 72)

    result = subprocess.run(cmd, capture_output=False)
    full_ok = (result.returncode == 0)
    print("\n" + "=" * 72)
    print(f"{'✅ FULL COMPLETATO' if full_ok else '❌ FULL FAILED (training abortito o errore)'}")


In [ ]:
# ===========================================================
# CELLA 6 — Display grafici (universal, self-contained)
# ===========================================================
import os, glob
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown

ckpt_dir = f"checkpoints/{TAG}"
plots_dir = f"{ckpt_dir}/plots"

if not os.path.isdir(plots_dir):
    print(f"❌ Cartella {plots_dir} non esistente.")
    print(f"   Cartelle disponibili:")
    for d in sorted(os.listdir("checkpoints")):
        print(f"     - {d}")
else:
    plots = sorted(glob.glob(f"{plots_dir}/*.png"))
    print(f"📊 {len(plots)} grafici generati per {TAG}:\n")
    for p in plots:
        display(Markdown(f"### {os.path.basename(p)}"))
        display(Image(filename=p))

In [ ]:
# ===========================================================
# CELLA 7 — Analisi numerica (universal, self-contained)
# ===========================================================
import os
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ckpt_dir = f"checkpoints/{TAG}"

# Per-epoch
epoch_csv = f"{ckpt_dir}/training_log.csv"
epoch_df = None
if os.path.isfile(epoch_csv):
    epoch_df = pd.read_csv(epoch_csv)
    if len(epoch_df) > 0:
        display(Markdown("### 📈 Per-epoch summary"))
        display(epoch_df.round(4))

# Per-batch (sempre presente grazie alla telemetria T)
batch_csv = f"{ckpt_dir}/training_batch_log.csv"
if os.path.isfile(batch_csv):
    batch_df = pd.read_csv(batch_csv)
    display(Markdown(f"### 📈 Per-batch summary (N={len(batch_df)})"))
    
    # IMPORTANTE: usa np.nan, NON pd.NA (bug pandas Azure)
    gn_pre = batch_df['gn_total_preclip'].replace([np.inf, -np.inf], np.nan)
    
    n_inf = int(batch_df['is_inf_grad'].sum())
    n_nan = int(batch_df['is_nan_loss'].sum())
    first_inf_list = batch_df.index[batch_df['is_inf_grad'] == 1].tolist()
    first_inf = int(first_inf_list[0]) + 1 if first_inf_list else None
    
    if epoch_df is not None and len(epoch_df) > 0:
        print(f"   • Best val_loss:          {epoch_df['val_total'].min():.5f}")
    print(f"   • Batch totali:           {len(batch_df)}")
    print(f"   • Inf grad batches:       {n_inf}/{len(batch_df)} "
          f"{'❌' if n_inf > 0 else '✅'}")
    if first_inf:
        print(f"   • Primo inf grad a:       B{first_inf}")
    print(f"   • NaN loss batches:       {n_nan}/{len(batch_df)} "
          f"{'❌' if n_nan > 0 else '✅'}")
    print(f"   • Spike rate medio:       {batch_df['spike_rate'].mean()*100:.2f}% "
          f"{'✅' if 0.05 <= batch_df['spike_rate'].mean() <= 0.30 else '⚠️'} (target 10-25%)")
    print(f"   • gn pre-clip mediana:    {batch_df['gn_total_preclip'].median():.4f}")
    print(f"   • gn pre-clip max finito: {gn_pre.max():.3e}")
    
    # Decomposizione per-layer al primo inf (se presente)
    if first_inf:
        print(f"\n   🔍 Per-layer gn al primo inf (B{first_inf}):")
        row = batch_df.iloc[first_inf - 1]
        layer_cols = ['gn_hidden_fc', 'gn_hidden_recU', 'gn_hidden_recV',
                      'gn_hidden_base_threshold', 'gn_hidden_thresh_jump', 'gn_out_fc']
        for c in layer_cols:
            v = row[c]
            mark = ' ❌ INF' if np.isinf(v) else (' ⚠️ NaN (no grad)' if np.isnan(v) else '')
            print(f"      {c:<32} {v:.3e}{mark}")

In [ ]:
# ===========================================================
# CELLA 8 — Estrai + commit + push results/<TAG>/ (universal)
# Funziona anche su training FAILED (artefatti dal batch_log persistono)
# ===========================================================
if not PUSH_RESULTS:
    print("⏭️  PUSH_RESULTS=False — skipped")
else:
    import os, shutil, glob, datetime
    import numpy as np
    import pandas as pd
    
    src_dir = f"checkpoints/{TAG}"
    dst_dir = f"results/{TAG}"
    
    if not os.path.isdir(src_dir):
        print(f"❌ {src_dir} non esistente — niente da pushare.")
    else:
        # Copia CSV + JSON + PNG (NO .pt: pesi restano in checkpoints/)
        if os.path.isdir(dst_dir):
            shutil.rmtree(dst_dir)
        os.makedirs(f"{dst_dir}/plots", exist_ok=True)
        
        for f in glob.glob(f"{src_dir}/*.csv") + glob.glob(f"{src_dir}/*.json"):
            shutil.copy2(f, dst_dir)
        for f in glob.glob(f"{src_dir}/plots/*.png"):
            shutil.copy2(f, f"{dst_dir}/plots/")
        
        # Crash info se presente
        crash_pt = f"{src_dir}/crash_model.pt"
        if os.path.isfile(crash_pt):
            import torch
            ck = torch.load(crash_pt, map_location='cpu', weights_only=False)
            with open(f"{dst_dir}/CRASH_INFO.txt", 'w') as f:
                f.write(f"Crash detected\nEpoch: {ck.get('epoch','?')}\nval_loss: {ck.get('val_loss','?')}\n")
        
        # Analisi per il commit message
        batch_csv = f"{dst_dir}/training_batch_log.csv"
        msg_extra = ""
        if os.path.isfile(batch_csv):
            bdf = pd.read_csv(batch_csv)
            gn_pre = bdf['gn_total_preclip'].replace([np.inf, -np.inf], np.nan)
            first_inf_list = bdf.index[bdf['is_inf_grad'] == 1].tolist()
            first_inf = int(first_inf_list[0]) + 1 if first_inf_list else None
            status = "FAILED" if first_inf else "OK"
            msg_extra = (
                f"\nStatus:         {status}\n"
                f"Batch totali:   {len(bdf)}\n"
                f"Primo inf grad: {f'B{first_inf}' if first_inf else 'nessuno'}\n"
                f"Inf totali:     {int(bdf['is_inf_grad'].sum())}\n"
                f"gn max finito:  {gn_pre.max():.3e}\n"
                f"Spike rate:     {bdf['spike_rate'].mean()*100:.2f}%\n"
            )
        
        ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
        commit_msg = (
            f"results: {TAG} ({ts})\n\n"
            f"Config: scheduler={CONFIG['scheduler']}, max_lr={CONFIG.get('max_lr','-')}, "
            f"seq_len={CONFIG['seq_len']}, epochs={CONFIG['epochs']}\n"
            f"{msg_extra}\n"
            f"Artefatti: CSV (per-epoca + per-batch) + plots G1-G13.\n"
            f"Pesi modello restano in checkpoints/{TAG}/ su Azure.\n"
        )
        with open('/tmp/commit_msg.txt', 'w', encoding='utf-8') as f:
            f.write(commit_msg)
        
        !git add {dst_dir}
        !git commit -F /tmp/commit_msg.txt
        !rm /tmp/commit_msg.txt
        !git push origin main
        print(f"\n✅ Push completato. In locale: git pull && ls results/{TAG}/")

In [ ]:
# ===========================================================
# CELLA 9 — Comparazione fra esperimenti (universal, self-contained)
# Confronta tutti i tag in results/ con metriche chiave.
# ===========================================================
import os, glob, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, HTML

RESULTS_DIR = "results"

if not os.path.isdir(RESULTS_DIR):
    print(f"❌ Cartella {RESULTS_DIR}/ inesistente. Esegui Cella 8 di almeno un esperimento.")
else:
    rows = []
    for tag_dir in sorted(os.listdir(RESULTS_DIR)):
        tag_path = os.path.join(RESULTS_DIR, tag_dir)
        if not os.path.isdir(tag_path):
            continue
        
        epoch_csv = f"{tag_path}/training_log.csv"
        batch_csv = f"{tag_path}/training_batch_log.csv"
        config_json = f"{tag_path}/config_snapshot.json"
        crash_file  = f"{tag_path}/CRASH_INFO.txt"
        
        # ── Estrai config ────────────────────────────────────────
        cfg = {}
        if os.path.isfile(config_json):
            with open(config_json) as f:
                cfg = json.load(f)
        
        # ── Estrai metriche per-epoca ────────────────────────────
        best_val   = np.nan
        n_epochs   = 0
        final_lr   = np.nan
        time_total = 0.0
        if os.path.isfile(epoch_csv):
            edf = pd.read_csv(epoch_csv)
            if len(edf) > 0:
                best_val   = edf['val_total'].min()
                n_epochs   = len(edf)
                final_lr   = edf['lr'].iloc[-1]
                time_total = edf['time_s'].sum()
        
        # ── Estrai metriche per-batch ────────────────────────────
        n_batch    = 0
        n_inf      = 0
        n_nan      = 0
        first_inf  = None
        spike_avg  = np.nan
        gn_max     = np.nan
        gn_median  = np.nan
        if os.path.isfile(batch_csv):
            bdf = pd.read_csv(batch_csv)
            n_batch   = len(bdf)
            n_inf     = int(bdf['is_inf_grad'].sum())
            n_nan     = int(bdf['is_nan_loss'].sum())
            spike_avg = bdf['spike_rate'].mean() * 100
            gn_pre    = bdf['gn_total_preclip'].replace([np.inf, -np.inf], np.nan)
            gn_max    = gn_pre.max()
            gn_median = gn_pre.median()
            first_inf_list = bdf.index[bdf['is_inf_grad'] == 1].tolist()
            if first_inf_list:
                first_inf = int(first_inf_list[0]) + 1
        
        # ── Status ───────────────────────────────────────────────
        if os.path.isfile(crash_file):
            status = "❌ CRASH"
        elif n_inf > 0:
            status = "⚠️ INF"
        elif n_epochs >= cfg.get('epochs', 0) and n_epochs > 0:
            status = "✅ OK"
        else:
            status = "❓ ?"
        
        rows.append({
            'TAG':         tag_dir,
            'Status':      status,
            'Sched':       cfg.get('scheduler', '?'),
            'max_lr':      cfg.get('max_lr', cfg.get('lr', np.nan)),
            'seq_len':     cfg.get('seq_len', np.nan),
            'epochs':      f"{n_epochs}/{cfg.get('epochs', '?')}",
            'best_val':    best_val,
            'spike%':      spike_avg,
            'gn_med':      gn_median,
            'gn_max':      gn_max,
            'n_inf':       n_inf,
            '1st_inf':     first_inf if first_inf else '-',
            'time_s':      time_total if time_total > 0 else np.nan,
        })
    
    if not rows:
        print(f"❌ Nessun esperimento in {RESULTS_DIR}/")
    else:
        df = pd.DataFrame(rows)
        
        # Format numerico
        for col, fmt in [('max_lr', '{:.1e}'), ('best_val', '{:.4f}'),
                         ('spike%', '{:.2f}'), ('gn_med', '{:.2e}'),
                         ('gn_max', '{:.2e}'), ('time_s', '{:.0f}')]:
            df[col] = df[col].apply(lambda x: fmt.format(x) if pd.notna(x)
                                    and not isinstance(x, str) else '-')
        
        display(Markdown(f"### 📊 Comparazione esperimenti (N={len(df)})"))
        display(df.to_html(index=False, escape=False,
                            classes='table table-striped', border=0))
        
        # ── Highlights: identifica il "best" per ciascuna metrica ──
        df_num = pd.DataFrame(rows)   # versione non formattata
        display(Markdown("### 🏆 Best per metrica"))
        
        # Best val_loss (solo run completati senza crash)
        ok_runs = df_num[df_num['Status'] == '✅ OK']
        if len(ok_runs) > 0:
            best_val_row = ok_runs.loc[ok_runs['best_val'].idxmin()]
            print(f"   🥇 Miglior val_loss:      {best_val_row['TAG']:<35} "
                  f"val={best_val_row['best_val']:.4f}")
        
        # Best spike rate (più vicino al target 15%)
        if len(df_num) > 0:
            df_num['spike_dist'] = (df_num['spike%'] - 15).abs()
            best_spike_row = df_num.loc[df_num['spike_dist'].idxmin()]
            print(f"   🎯 Spike rate piu' sano:  {best_spike_row['TAG']:<35} "
                  f"spike={best_spike_row['spike%']:.2f}% (target 15%)")
        
        # Worst gn_max (esplosione peggiore)
        if df_num['gn_max'].notna().any():
            worst_gn_row = df_num.loc[df_num['gn_max'].idxmax()]
            print(f"   💥 Esplosione peggiore:   {worst_gn_row['TAG']:<35} "
                  f"gn_max={worst_gn_row['gn_max']:.2e}")
        
        # Più stabile (gn_median più bassa)
        if df_num['gn_median'].notna().any():
            stable_row = df_num.loc[df_num['gn_median'].idxmin()]
            print(f"   📐 Più stabile (gn med):  {stable_row['TAG']:<35} "
                  f"gn_med={stable_row['gn_median']:.2e}")
        
        # Tasso di crash
        n_total = len(df_num)
        n_crash = (df_num['Status'].str.startswith('❌')).sum()
        n_inf   = (df_num['Status'] == '⚠️ INF').sum()
        n_ok    = (df_num['Status'] == '✅ OK').sum()
        print(f"\n📈 Statistiche aggregate:")
        print(f"   Totale esperimenti:  {n_total}")
        print(f"   ✅ OK:               {n_ok} ({n_ok/n_total*100:.0f}%)")
        print(f"   ⚠️  Con inf grad:    {n_inf} ({n_inf/n_total*100:.0f}%)")
        print(f"   ❌ Crash:            {n_crash} ({n_crash/n_total*100:.0f}%)")